In [52]:
import pandas as pd 

In [53]:
data = pd.read_csv('../dataset/cleaned_data.csv')
data.head()

,satisfaction_level,last_evaluation,number_project,average_montly_hours,time_spend_company,Work_accident,left,promotion_last_5years,Department,salary
0,0.38,0.53,2,157,3,0,1,0,sales,low
1,0.80,0.86,5,262,6,0,1,0,sales,medium
2,0.11,0.88,7,272,4,0,1,0,sales,medium
3,0.72,0.87,5,223,5,0,1,0,sales,low
4,0.37,0.52,2,159,3,0,1,0,sales,low


In [54]:
X = data.drop('left',axis=1).copy()

In [55]:
y = data['left'].copy()

In [56]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,stratify=y,random_state=42)

In [57]:
categorical_features = ['Department','salary']
numerical_features = [col for col in X.columns.unique() if col not in categorical_features]
numerical_features

['satisfaction_level',
 'last_evaluation',
 'number_project',
 'average_montly_hours',
 'time_spend_company',
 'Work_accident',
 'promotion_last_5years']

In [58]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
numerical_transformer = Pipeline(steps=[
    ('scaler',StandardScaler())])
categorical_transformer = Pipeline(steps=[
    ('encode',OneHotEncoder(handle_unknown='ignore'))
])

In [59]:
data['left'].value_counts()
## Imbalanced class but mild ratio 

left
0    10000
1     1991
Name: count, dtype: int64

In [60]:
from sklearn.compose import ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ('numeric',numerical_transformer,numerical_features),
        ('categorical',categorical_transformer,categorical_features) 
    ]
)

In [61]:
from sklearn.linear_model import LogisticRegression 
logreg_pipe = Pipeline(steps=[
    ('preprocessor',preprocessor),
    ('model',LogisticRegression(class_weight='balanced',random_state=42))
])
logreg_pipe.fit(X_train,y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('numeric',
                                                  Pipeline(steps=[('scaler',
                                                                   StandardScaler())]),
                                                  ['satisfaction_level',
                                                   'last_evaluation',
                                                   'number_project',
                                                   'average_montly_hours',
                                                   'time_spend_company',
                                                   'Work_accident',
                                                   'promotion_last_5years']),
                                                 ('categorical',
                                                  Pipeline(steps=[('encode',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['Department', 'salary'])])),
                ('model',
                 LogisticRegression(class_weight='balanced', random_state=42))])

In [62]:
from sklearn.metrics import classification_report
predictions = logreg_pipe.predict(X_test)

In [63]:
print(classification_report(y_test,predictions, target_names=['Stayed', 'Left']))

              precision    recall  f1-score   support

      Stayed       0.96      0.78      0.86      2001
        Left       0.43      0.84      0.57       398

    accuracy                           0.79      2399
   macro avg       0.70      0.81      0.71      2399
weighted avg       0.87      0.79      0.81      2399

